# 01 — Download Raw Data
**New York Quarterly Emergency Department Demand Forecasting**

This notebook contains only the **data acquisition layer** from the original capstone workflow.

It downloads and preserves the raw public-source files used by the revised project:

- SPARCS ED Encounters by Facility
- Census Gazetteer county geography
- ACS 5-Year county demographics
- Open-Meteo historical daily weather

The revised project now separates responsibilities:

**This notebook:** download and preserve raw data  
**Oracle SQL:** cleaning, aggregation, joins, feature engineering, and QA  
**Python modeling notebook:** EDA, model training, evaluation, and model export  
**Power BI:** interactive reporting and dashboarding

No SQL-equivalent cleaning, aggregation, joins, or machine-learning work is performed here.


## 0 · Local setup

This notebook is intended to run locally in **VS Code/Jupyter** from:

`C:\Users\emily\DATA_SCIENCE\PROJECT\ed_demand_app`

Expected structure:

```text
ed_demand_app/
├── data/
│   └── raw/
├── notebooks/
│   └── 01_download_raw_data.ipynb
├── sql/
├── model/
└── ...
```

The path logic below works whether Jupyter starts in the project root or in `notebooks/`. It falls back to your current Windows project path when needed.


In [ ]:
from pathlib import Path
import io
import os
import re
import time
import zipfile

import numpy as np
import pandas as pd

try:
    import requests
except ImportError as exc:
    raise ImportError(
        "The 'requests' package is required. "
        "Install it with: pip install requests"
    ) from exc

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Resolve the local project root.
cwd = Path.cwd().resolve()

if cwd.name.lower() == "notebooks":
    PROJECT_DIR = cwd.parent
elif (cwd / "notebooks").exists() and (cwd / "sql").exists():
    PROJECT_DIR = cwd
else:
    LOCAL_PROJECT = Path(
        r"C:\Users\emily\DATA_SCIENCE\PROJECT\ed_demand_app"
    )
    if LOCAL_PROJECT.exists():
        PROJECT_DIR = LOCAL_PROJECT
    else:
        raise FileNotFoundError(
            "Could not identify the ed_demand_app project root. "
            "Open the ed_demand_app folder in VS Code, then run this notebook."
        )

RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Raw-data folder:", RAW_DIR)


## 1 · Configuration

The quarterly project keeps the same acquisition logic as the original notebook:

- **Geography:** New York facility county
- **Target source:** SPARCS facility-quarter ED encounters
- **ACS alignment:** two-year lag relative to the target year
- **Weather alignment:** same quarter from the previous year

The acquisition notebook determines which ACS and weather source years are required from the available SPARCS target years.

### Refresh behavior

- `REFRESH_FACILITY_SOURCE = False`  
  Reuse the saved SPARCS raw snapshot when it already exists.

- `FORCE_REBUILD_RAW_FILES = False`  
  Reuse saved geography/ACS/weather files and only add missing periods.

Set either option to `True` only when you intentionally want to refresh/rebuild raw data.


In [ ]:
STATE_USPS = "NY"
STATE_FIPS = "36"

REFRESH_FACILITY_SOURCE = False
FORCE_REBUILD_RAW_FILES = False

SOCRATA_DOMAIN = "health.data.ny.gov"
SPARCS_RESOURCE_ID = "5gzv-zv2z"

CENSUS_DATASET = "acs/acs5"

OPEN_METEO_ARCHIVE = (
    "https://archive-api.open-meteo.com/v1/archive"
)

GAZETTEER_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/"
    "2023_Gazetteer/2023_Gaz_counties_national.zip"
)

CENSUS_VARIABLES = {
    "pop": "B01003_001E",
    "median_income": "B19013_001E",
    "pov_total": "B17001_001E",
    "pov_below": "B17001_002E",
    "age_total": "B01001_001E",
    "male_under5": "B01001_003E",
    "female_under5": "B01001_027E",
}

AGE_65PLUS_VARS = [
    "B01001_020E",
    "B01001_021E",
    "B01001_022E",
    "B01001_023E",
    "B01001_024E",
    "B01001_025E",
    "B01001_044E",
    "B01001_045E",
    "B01001_046E",
    "B01001_047E",
    "B01001_048E",
    "B01001_049E",
]

# Optional. Keep the key outside the notebook if you use one.
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")

print("State:", STATE_USPS)
print("SPARCS resource:", SPARCS_RESOURCE_ID)


## 2 · Acquisition helper functions

Only helpers required for downloading and preserving the raw sources are retained from the original notebook.

Data-cleaning helpers used later for modeling have intentionally been removed because **Oracle SQL now owns the transformation layer**.


In [ ]:
def clean_column_names(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return out


def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )


def name_key(value):
    text = str(value).lower().split(",")[0]
    text = text.replace("county", "").replace(".", "").replace("'", "")
    return re.sub(r"\s+", " ", text).strip()


def download_socrata(resource_id, limit=50000):
    url = f"https://{SOCRATA_DOMAIN}/resource/{resource_id}.json"
    frames = []
    offset = 0

    while True:
        response = requests.get(
            url,
            params={"$limit": limit, "$offset": offset},
            timeout=90,
        )
        response.raise_for_status()
        rows = response.json()

        if not rows:
            break

        frames.append(pd.DataFrame(rows))

        if len(rows) < limit:
            break

        offset += limit
        time.sleep(0.2)

    if not frames:
        raise RuntimeError("The Socrata request returned no rows.")

    return pd.concat(frames, ignore_index=True)


# Phase 1 — Acquire raw source files

## 3 · SPARCS ED Encounters by Facility

Download the SPARCS ED Encounters by Facility source and preserve it at its original facility-quarter/statewide-summary grain.

The SQL pipeline will later clean numeric encounter fields, remove non-county rows for modeling, aggregate facilities to county-quarter, and validate suppression/missingness.

This notebook does **not** perform those transformations.


In [ ]:
FACILITY_RAW_PATH = RAW_DIR / "sparcs_ed_facility_quarter.csv"


def load_facility_source():
    should_download = (
        REFRESH_FACILITY_SOURCE
        or FORCE_REBUILD_RAW_FILES
        or not FACILITY_RAW_PATH.exists()
    )

    if should_download:
        try:
            raw = download_socrata(SPARCS_RESOURCE_ID)
            raw.to_csv(FACILITY_RAW_PATH, index=False)
            print("Downloaded current SPARCS facility source.")
        except Exception as error:
            if not FACILITY_RAW_PATH.exists():
                raise
            print("SPARCS refresh failed:", error)
            print("Using the existing local SPARCS raw file.")
            raw = pd.read_csv(FACILITY_RAW_PATH, dtype=str)
    else:
        raw = pd.read_csv(FACILITY_RAW_PATH, dtype=str)
        print("Loaded existing local SPARCS raw file.")

    checked = clean_column_names(raw)

    required = {
        "year",
        "quarter",
        "facility_id",
        "facility_name",
        "total_ed_encounters",
        "facility_county_fips",
    }
    missing = required - set(checked.columns)

    if missing:
        raise KeyError(f"SPARCS is missing columns: {sorted(missing)}")

    actual_facility_rows = (
        checked["facility_county_fips"].notna()
        & checked["facility_id"].astype(str).ne("0")
    )

    if actual_facility_rows.sum() == 0:
        raise ValueError("No facility-county rows were found.")

    return checked


facility_raw = load_facility_source()

facility_quarter_rows = (
    facility_raw["facility_county_fips"].notna()
    & facility_raw["facility_id"].astype(str).ne("0")
    & facility_raw["quarter"].astype(str).str.upper().isin(
        ["Q1", "Q2", "Q3", "Q4"]
    )
)

facility_years = (
    to_num(facility_raw.loc[facility_quarter_rows, "year"])
    .dropna()
    .astype(int)
)

FACILITY_START_YEAR = int(facility_years.min())
FACILITY_END_YEAR = int(facility_years.max())

# ACS uses a conservative two-year lag; weather uses a one-year lag.
ACS_SOURCE_YEARS = list(
    range(FACILITY_START_YEAR - 2, FACILITY_END_YEAR - 1)
)
WEATHER_SOURCE_YEARS = list(
    range(FACILITY_START_YEAR - 1, FACILITY_END_YEAR)
)

DATA_AS_OF_DATE = pd.Timestamp.fromtimestamp(
    FACILITY_RAW_PATH.stat().st_mtime
).date().isoformat()

print("SPARCS rows:", len(facility_raw))
print("Facility target years:", FACILITY_START_YEAR, "to", FACILITY_END_YEAR)
print("ACS source years:", min(ACS_SOURCE_YEARS), "to", max(ACS_SOURCE_YEARS))
print(
    "Weather source years:",
    min(WEATHER_SOURCE_YEARS),
    "to",
    max(WEATHER_SOURCE_YEARS),
)
print("SPARCS snapshot date:", DATA_AS_OF_DATE)

preview_columns = [
    "year",
    "quarter",
    "facility_id",
    "facility_name",
    "facility_county_fips",
    "total_ed_encounters",
]
if "facility_county" in facility_raw.columns:
    preview_columns.insert(5, "facility_county")

display(
    facility_raw.loc[
        facility_quarter_rows,
        preview_columns,
    ].head()
)


## 4 · New York county geography

Download New York county FIPS codes and county centroids from the Census Gazetteer.

Output:

`data/raw/ny_county_geo.csv`

FIPS is the common geographic join key used later in Oracle SQL.


In [ ]:
COUNTY_GEO_PATH = RAW_DIR / "ny_county_geo.csv"

if COUNTY_GEO_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    county_geo = pd.read_csv(
        COUNTY_GEO_PATH,
        dtype={"fips": str},
    )
    print("Loaded existing county geography file.")
else:
    response = requests.get(GAZETTEER_URL, timeout=90)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        text_name = next(
            name for name in zf.namelist()
            if name.lower().endswith(".txt")
        )
        geo_raw = pd.read_csv(
            io.StringIO(zf.read(text_name).decode("latin-1")),
            sep="\t",
            dtype=str,
        )

    geo_raw.columns = [column.strip() for column in geo_raw.columns]
    geo_raw = geo_raw.loc[geo_raw["USPS"].eq(STATE_USPS)].copy()

    county_geo = pd.DataFrame({
        "fips": geo_raw["GEOID"].str.strip(),
        "county_name": geo_raw["NAME"].str.strip(),
        "lat": geo_raw["INTPTLAT"].str.strip().astype(float),
        "lon": geo_raw["INTPTLONG"].str.strip().astype(float),
    })

    county_geo["name_key"] = county_geo["county_name"].map(name_key)

    county_geo.to_csv(
        COUNTY_GEO_PATH,
        index=False,
    )
    print("Downloaded and saved county geography.")

county_geo["fips"] = county_geo["fips"].astype(str).str.zfill(5)

print("New York counties:", len(county_geo))
display(county_geo.head())


## 5 · Census ACS demographics

Download ACS 5-Year county estimates for the source years required by the SPARCS target period.

Output:

`data/raw/census_acs_county.csv`

The raw ACS API fields are preserved. Oracle SQL later performs numeric cleaning and demographic feature engineering.


In [ ]:
CENSUS_RAW_PATH = RAW_DIR / "census_acs_county.csv"


def download_acs_year(year):
    variables = list(CENSUS_VARIABLES.values()) + AGE_65PLUS_VARS

    url = f"https://api.census.gov/data/{year}/{CENSUS_DATASET}"

    params = {
        "get": "NAME," + ",".join(variables),
        "for": "county:*",
        "in": f"state:{STATE_FIPS}",
    }

    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    response = requests.get(
        url,
        params=params,
        timeout=60,
    )

    if response.status_code != 200:
        print(f"ACS {year} is not available.")
        return pd.DataFrame()

    rows = response.json()
    frame = pd.DataFrame(rows[1:], columns=rows[0])
    frame["year"] = year
    return frame


if CENSUS_RAW_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    census_raw = pd.read_csv(
        CENSUS_RAW_PATH,
        dtype={"state": str, "county": str},
    )
    print("Loaded existing ACS raw file.")
else:
    census_raw = pd.DataFrame()


saved_acs_years = set()

if not census_raw.empty and "year" in census_raw:
    saved_acs_years = set(
        to_num(census_raw["year"]).dropna().astype(int)
    )


missing_acs_years = [
    year
    for year in ACS_SOURCE_YEARS
    if year not in saved_acs_years
]

print(
    "Missing ACS years to download:",
    missing_acs_years if missing_acs_years else "none",
)


new_acs_frames = [
    download_acs_year(year)
    for year in missing_acs_years
]
new_acs_frames = [
    frame
    for frame in new_acs_frames
    if not frame.empty
]


if new_acs_frames:
    census_raw = pd.concat(
        [census_raw] + new_acs_frames,
        ignore_index=True,
    )


if census_raw.empty:
    raise RuntimeError("No ACS data are available.")


# Keep stable identifier formatting and avoid duplicate raw API rows.
census_raw["state"] = census_raw["state"].astype(str).str.zfill(2)
census_raw["county"] = census_raw["county"].astype(str).str.zfill(3)
census_raw["year"] = to_num(census_raw["year"]).astype("Int64")

census_raw = (
    census_raw
    .drop_duplicates(["state", "county", "year"], keep="last")
    .sort_values(["year", "state", "county"])
    .reset_index(drop=True)
)

census_raw.to_csv(
    CENSUS_RAW_PATH,
    index=False,
)

print(
    "ACS years:",
    int(census_raw["year"].min()),
    "to",
    int(census_raw["year"].max()),
)
print("ACS rows:", len(census_raw))


## 6 · Daily county weather

Download daily historical weather at each New York county centroid.

Output:

`data/raw/weather_daily_by_county.csv`

The daily grain is intentionally preserved here. Oracle SQL later aggregates daily observations to county-quarter and applies the prior-year alignment.


In [ ]:
WEATHER_RAW_PATH = RAW_DIR / "weather_daily_by_county.csv"


def download_weather(row, start_date, end_date):
    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "daily": (
            "temperature_2m_max,temperature_2m_min,"
            "precipitation_sum,snowfall_sum"
        ),
        "temperature_unit": "fahrenheit",
        "precipitation_unit": "inch",
        "timezone": "America/New_York",
    }

    response = requests.get(
        OPEN_METEO_ARCHIVE,
        params=params,
        timeout=60,
    )
    response.raise_for_status()
    daily = response.json()["daily"]

    return pd.DataFrame({
        "fips": str(row["fips"]).zfill(5),
        "date": daily["time"],
        "tmax": daily["temperature_2m_max"],
        "tmin": daily["temperature_2m_min"],
        "precip": daily["precipitation_sum"],
        "snowfall": daily.get(
            "snowfall_sum",
            [np.nan] * len(daily["time"]),
        ),
    })


if WEATHER_RAW_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    weather_daily = pd.read_csv(
        WEATHER_RAW_PATH,
        dtype={"fips": str},
    )
    weather_daily["date"] = pd.to_datetime(
        weather_daily["date"],
        errors="coerce",
    )
    print("Loaded existing weather raw file.")
else:
    weather_daily = pd.DataFrame(
        columns=[
            "fips",
            "date",
            "tmax",
            "tmin",
            "precip",
            "snowfall",
        ]
    )


required_start = pd.Timestamp(
    f"{min(WEATHER_SOURCE_YEARS)}-01-01"
)
required_end = pd.Timestamp(
    f"{max(WEATHER_SOURCE_YEARS)}-12-31"
)

weather_additions = []


for row_number, row in county_geo.iterrows():
    fips = str(row["fips"]).zfill(5)

    county_saved = weather_daily.loc[
        weather_daily["fips"]
        .astype(str)
        .str.zfill(5)
        .eq(fips)
    ]

    if county_saved.empty:
        start_date = required_start
    else:
        start_date = max(
            required_start,
            county_saved["date"].max() + pd.Timedelta(days=1),
        )

    if start_date > required_end:
        continue

    print(
        f"Weather {row_number + 1}/{len(county_geo)} | "
        f"{fips} | {start_date.date()} to {required_end.date()}"
    )

    weather_additions.append(
        download_weather(
            row,
            str(start_date.date()),
            str(required_end.date()),
        )
    )

    time.sleep(1)


if weather_additions:
    weather_daily = pd.concat(
        [
            weather_daily,
            pd.concat(
                weather_additions,
                ignore_index=True,
            ),
        ],
        ignore_index=True,
    )


if weather_daily.empty:
    raise RuntimeError("No weather data are available.")


weather_daily["fips"] = (
    weather_daily["fips"]
    .astype(str)
    .str.zfill(5)
)

weather_daily["date"] = pd.to_datetime(
    weather_daily["date"],
    errors="coerce",
)

weather_daily = (
    weather_daily
    .dropna(subset=["fips", "date"])
    .drop_duplicates(["fips", "date"], keep="last")
    .sort_values(["fips", "date"])
    .reset_index(drop=True)
)


weather_daily.assign(
    date=weather_daily["date"].dt.strftime("%Y-%m-%d")
).to_csv(
    WEATHER_RAW_PATH,
    index=False,
)


print(
    "Weather dates:",
    weather_daily["date"].min().date(),
    "to",
    weather_daily["date"].max().date(),
)
print("Weather rows:", len(weather_daily))


## 7 · Verify the raw-data folder

This confirms that the four sources acquired by the original quarterly notebook exist locally before Oracle import.

### About `sparcs_ppv_by_county.csv`

The uploaded original quarterly notebook does **not** contain PPV acquisition code, so this revised notebook does not invent a PPV download method.

If you already have the legacy file:

`data/raw/sparcs_ppv_by_county.csv`

keep it there for provenance/reference. The current SQL modeling pipeline preserves it but does not use it in `FINAL_COUNTY_QUARTER_ANALYSIS`.


In [ ]:
expected_files = [
    RAW_DIR / "sparcs_ed_facility_quarter.csv",
    RAW_DIR / "ny_county_geo.csv",
    RAW_DIR / "census_acs_county.csv",
    RAW_DIR / "weather_daily_by_county.csv",
]

optional_ppv = RAW_DIR / "sparcs_ppv_by_county.csv"

summary_rows = []

for file_path in expected_files:
    summary_rows.append({
        "file": file_path.name,
        "required": True,
        "exists": file_path.exists(),
        "size_mb": (
            round(file_path.stat().st_size / 1024**2, 2)
            if file_path.exists()
            else None
        ),
    })

summary_rows.append({
    "file": optional_ppv.name,
    "required": False,
    "exists": optional_ppv.exists(),
    "size_mb": (
        round(optional_ppv.stat().st_size / 1024**2, 2)
        if optional_ppv.exists()
        else None
    ),
})

raw_file_summary = pd.DataFrame(summary_rows)
display(raw_file_summary)

missing_required = raw_file_summary.loc[
    raw_file_summary["required"]
    & ~raw_file_summary["exists"],
    "file",
].tolist()

if missing_required:
    raise FileNotFoundError(
        "Missing required raw files: "
        + ", ".join(missing_required)
    )

print("All required raw acquisition files are ready.")


# Next step — Oracle SQL

Import the required files into Oracle as:

```text
sparcs_ed_facility_quarter.csv
    → RAW_SPARCS_ED_FACILITY_QUARTER

census_acs_county.csv
    → RAW_CENSUS_ACS_COUNTY

ny_county_geo.csv
    → RAW_NY_COUNTY_GEO

weather_daily_by_county.csv
    → RAW_WEATHER_DAILY_BY_COUNTY
```

If you preserve the optional legacy PPV file:

```text
sparcs_ppv_by_county.csv
    → RAW_SPARCS_PPV_BY_COUNTY
```

Then run:

`sql/01_build_county_quarter_analysis.sql`

After SQL validation, export:

`FINAL_COUNTY_QUARTER_ANALYSIS`

to:

`data/processed/county_quarter_analysis.csv`

Then run:

`notebooks/02_ed_demand_modeling.ipynb`
